首先安装 python3.9 和 mindspore、mindnlp。

In [ ]:
%%capture captured_output

# 安装python 3.9版本的kernel
!/home/ma-user/anaconda3/bin/conda create -n python-3.9.0 python=3.9.0 -y --override-channels --channel https://mirrors.tuna.tsinghua.edu.cn/anaconda/pkgs/main
!/home/ma-user/anaconda3/envs/python-3.9.0/bin/pip install ipykernel

In [ ]:
import json
import os

data = {
   "display_name": "python-3.9.0",
   "env": {
      "PATH": "/home/ma-user/anaconda3/envs/python-3.9.0/bin:/home/ma-user/anaconda3/envs/python-3.7.10/bin:/modelarts/authoring/notebook-conda/bin:/opt/conda/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/home/ma-user/modelarts/ma-cli/bin:/home/ma-user/modelarts/ma-cli/bin"
   },
   "language": "python",
   "argv": [
      "/home/ma-user/anaconda3/envs/python-3.9.0/bin/python",
      "-m",
      "ipykernel",
      "-f",
      "{connection_file}"
   ]
}

if not os.path.exists("/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/"):
    os.mkdir("/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/")

with open('/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/kernel.json', 'w') as f:
    json.dump(data, f, indent=4)

安装完成之后重启 kernel，选择 "python 3.9"。

---

在大模型平台运行代码：https://pangu.huaweicloud.com/gallery/asset-detail.html?id=3b23f58b-0c88-4777-9abe-3f06b05c57fd

预训练 Beit 模型时，我们首先将输入图片分块，然后使用 DALL-E 模型的 encoder 部分将块转换为离散的 token；接着随机遮盖一些块，将剩余的部分输入 Beit 模型的 Transformer，让其预测被遮盖部分的 token。

在以下的代码中，我们重现这一训练过程。

In [ ]:
%%capture captured_output
!pip uninstall mindspore-gpu -y
!pip install https://ms-release.obs.cn-north-4.myhuaweicloud.com/2.2.11/MindSpore/unified/x86_64/mindspore-2.2.11-cp39-cp39-linux_x86_64.whl --trusted-host ms-release.obs.cn-north-4.myhuaweicloud.com -i https://pypi.tuna.tsinghua.edu.cn/simple
!pip install download nltk -i https://pypi.tuna.tsinghua.edu.cn/simple
!pip install mindnlp

In [ ]:
!wget https://dalle-ckpt.obs.cn-north-4.myhuaweicloud.com/utils.py
!wget https://dalle-ckpt.obs.cn-north-4.myhuaweicloud.com/conv.py
!wget https://dalle-ckpt.obs.cn-north-4.myhuaweicloud.com/encoder.py
!wget https://dalle-ckpt.obs.cn-north-4.myhuaweicloud.com/masking_generator.py
!wget https://dalle-ckpt.obs.cn-north-4.myhuaweicloud.com/dalle.ckpt

In [ ]:
from PIL import Image
import requests

# 下载输入图像
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)

image

In [ ]:
from mindnlp.transformers import BeitForMaskedImageModeling

# 下载使用的 Beit 模型
model = BeitForMaskedImageModeling.from_pretrained("microsoft/beit-base-patch16-224-pt22k")

In [ ]:
from masking_generator import MaskingGenerator

window_size = model.beit.embeddings.patch_embeddings.patch_shape
num_masking_patches = 75
max_mask_patches_per_block = None
min_mask_patches_per_block = 16

# 生成遮盖图片的 mask
mask_generator = MaskingGenerator(
            window_size, num_masking_patches=num_masking_patches,
            max_num_patches=max_mask_patches_per_block,
            min_num_patches=min_mask_patches_per_block,
        )

Beit 模型的训练需要三个输入：pixel_values（原图分块后得到的 Tensor），pixel_values_dall_e（经过 dall_e 处理得到每块对应的 token），bool_masked_pos（随机遮盖某些块的 mask 矩阵）。

我们依次获取这三个输入。

In [ ]:
import mindspore as ms
from mindnlp.transformers import BeitImageProcessor

image_processor = BeitImageProcessor()

# 获得 pixel_values
pixel_values = image_processor(image, return_tensors="ms").pixel_values
pixel_values.shape

In [ ]:
from utils import map_pixels
from mindspore import dataset
from mindspore.dataset import transforms
# 获得 pixel_values_dall_e
visual_token_transform = transforms.Compose([
                transforms.vision.Resize((112,112), transforms.vision.Inter.LINEAR),
                dataset.vision.ToTensor(),
                map_pixels,
            ])
pixel_values_dall_e = visual_token_transform(image)[0]
pixel_values_dall_e.shape

In [ ]:
# 获得 mask
bool_masked_pos = mask_generator()
bool_masked_pos = ms.Tensor(bool_masked_pos).unsqueeze(0)

bool_masked_pos.shape

In [ ]:
import encoder

# 从 ckpt 文件中加载 encoder
enc = encoder.Encoder()
params = ms.load_checkpoint("dalle.ckpt")
ms.load_param_into_net(enc,params)

In [ ]:
pixel_values_dall_e = ms.Tensor(pixel_values_dall_e)
pixel_values_dall_e.shape

In [ ]:
# 计算输入图像每块对应的 token
z_logits = enc(pixel_values_dall_e)
input_ids = ms.ops.argmax(z_logits, dim=1).flatten(start_dim=1)
bool_masked_pos = bool_masked_pos.flatten(start_dim=1).to(ms.bool_)
labels = input_ids[bool_masked_pos]

In [ ]:
input_ids.shape

In [ ]:
# 由于 mask 生成的随机性，labels 的 shape 可能每次都不同
labels.shape

In [ ]:
outputs = model(pixel_values, bool_masked_pos)

In [ ]:
# labels 是 DALL-E 推理得出的被遮盖块的 token
labels

In [ ]:
predictions = outputs.logits[bool_masked_pos].argmax(-1)

In [ ]:
# predictions 是 Beit 预测的被遮盖块的 token
predictions

接着我们做一些集成测试。

In [ ]:
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)

# 准备数据（缩放+标准化）
mean = (0.5, 0.5, 0.5)
std = (0.5, 0.5, 0.5)
transform = transforms.Compose([transforms.vision.Resize((224, 224)),
                                dataset.vision.ToTensor(),
                                dataset.vision.Normalize(mean, std, is_hwc=False)])

pixel_values = ms.Tensor(transform(image)[0]).unsqueeze(0)
pixel_values.shape

In [ ]:
pixel_values[0,:3,:3,:3]

In [ ]:
bool_masked_pos = ms.ops.ones((1, 196), dtype=ms.bool_)

# 得到输出
outputs = model(pixel_values, bool_masked_pos)

In [ ]:
outputs.logits.shape

In [ ]:
outputs.logits[bool_masked_pos][:3,:3]